# Lab - Model, Effort and max_tokens

Three settings your application chooses for every request it sends to Claude. Choose
them per step, and check what comes back before a customer sees it.

| Task | What you learn |
| --- | --- |
| 1 | A model family for each step: Haiku for the log note, Sonnet for the chat |
| 2 | An effort level for each step: low for simple, high for tangled |
| 3 | `max_tokens` is a ceiling - read `stop_reason` and fix a cut reply first |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `FUELIX_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
client = Anthropic()

# One constant per model family, so a new version is a one-line change.
MODEL_CHAT = "claude-sonnet-5-5"      # the balanced default: the conversation
MODEL_SUMMARY = "claude-haiku-5-5"    # fast and inexpensive: simple, high-volume steps

SYSTEM = (
    "You are ShopAssist, an online store's support assistant. "
    "Returns are accepted within 30 days of delivery."
)

# A closed conversation, waiting for its two-line note in the log.
CLOSED_CONVERSATION = (
    "Customer: I bought a jacket 20 days ago. Can I still return it?\n"
    "ShopAssist: Yes - about 10 days are left in our 30-day return window."
)

---

## Task 1 - A model for each step

In [ ]:
# ============================================================
# TASK 1 - A model for each step
# ============================================================
#
# WHAT TO DO
#   Send two steps of ShopAssist to the right model family: the
#   two-line log note for a closed conversation, and an answer
#   to a customer.
#
# WHY IT MATTERS
#   It is tempting to pick one model for the whole application.
#   A better approach is to choose for each step, by its
#   difficulty, its speed and its volume. The log note is simple
#   and runs after every conversation, so Haiku fits. The
#   conversation needs policy judgment, so Sonnet runs it.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The first setting is the model".
#
# HOW TO DO IT
#   Two blanks, both constants from the setup cell:
#
#       MODEL_SUMMARY   Haiku - for the log note
#       MODEL_CHAT      Sonnet - for the customer
# ============================================================

# BEGIN SOLUTION
summary = client.messages.create(
    model=MODEL_SUMMARY,
    max_tokens=100,
    messages=[{"role": "user", "content":
               "Summarize this support conversation in two lines for the log.\n\n"
               + CLOSED_CONVERSATION}],
)

answer = client.messages.create(
    model=MODEL_CHAT,
    max_tokens=300,
    output_config={"effort": "low"},
    system=SYSTEM,
    messages=[{"role": "user", "content": "Has my order ORD-12345678 shipped?"}],
)
# SCAFFOLD: summary = client.messages.create(
# SCAFFOLD:     model=...,   # MODEL_SUMMARY - a simple step that runs after every conversation
# SCAFFOLD:     max_tokens=100,
# SCAFFOLD:     messages=[{"role": "user", "content":
# SCAFFOLD:                "Summarize this support conversation in two lines for the log.\n\n"
# SCAFFOLD:                + CLOSED_CONVERSATION}],
# SCAFFOLD: )
# SCAFFOLD:
# SCAFFOLD: answer = client.messages.create(
# SCAFFOLD:     model=...,   # MODEL_CHAT - the conversation needs policy judgment
# SCAFFOLD:     max_tokens=300,
# SCAFFOLD:     output_config={"effort": "low"},
# SCAFFOLD:     system=SYSTEM,
# SCAFFOLD:     messages=[{"role": "user", "content": "Has my order ORD-12345678 shipped?"}],
# SCAFFOLD: )
# END SOLUTION: replace each ... below with the constant named beside it

print("log note on", summary.model)
print(summary.content[0].text)
print()
print("answer on", answer.model)
print(answer.content[0].text)

check("model_per_step", summary=summary, answer=answer)

---

## Task 2 - An effort level for each step

In [ ]:
# ============================================================
# TASK 2 - An effort level for each step
# ============================================================
#
# WHAT TO DO
#   Send two customer questions to Sonnet, each with the effort
#   level it deserves.
#
# WHY IT MATTERS
#   Before Sonnet writes a reply it can first reason through the
#   problem on its own; current models decide for themselves when
#   and for how long. Your application does not control that
#   reasoning directly - it sets an effort level, which tells the
#   model how much work a step deserves. Confirming that an order
#   shipped needs little. Three orders that overlap, with a
#   partial refund on one, need more.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The second setting is effort".
#
# HOW TO DO IT
#   Two blanks, each an effort level in quotes:
#
#       "low"    the quick shipping question - fast answer
#       "high"   the tangled case - more work is worth it
# ============================================================

# BEGIN SOLUTION
quick = client.messages.create(
    model=MODEL_CHAT,
    max_tokens=300,
    output_config={"effort": "low"},
    system=SYSTEM,
    messages=[{"role": "user", "content": "Has my order ORD-12345678 shipped?"}],
)

tangled = client.messages.create(
    model=MODEL_CHAT,
    max_tokens=1000,
    output_config={"effort": "high"},
    system=SYSTEM,
    messages=[{"role": "user", "content":
               "I have three orders, ORD-12345678, ORD-87654321 and ORD-11112222, and one "
               "was partly refunded. Which of them can I still return?"}],
)
# SCAFFOLD: quick = client.messages.create(
# SCAFFOLD:     model=MODEL_CHAT,
# SCAFFOLD:     max_tokens=300,
# SCAFFOLD:     output_config={"effort": ...},   # "low" - a quick confirmation
# SCAFFOLD:     system=SYSTEM,
# SCAFFOLD:     messages=[{"role": "user", "content": "Has my order ORD-12345678 shipped?"}],
# SCAFFOLD: )
# SCAFFOLD:
# SCAFFOLD: tangled = client.messages.create(
# SCAFFOLD:     model=MODEL_CHAT,
# SCAFFOLD:     max_tokens=1000,
# SCAFFOLD:     output_config={"effort": ...},   # "high" - overlapping orders, a partial refund
# SCAFFOLD:     system=SYSTEM,
# SCAFFOLD:     messages=[{"role": "user", "content":
# SCAFFOLD:                "I have three orders, ORD-12345678, ORD-87654321 and ORD-11112222, and one "
# SCAFFOLD:                "was partly refunded. Which of them can I still return?"}],
# SCAFFOLD: )
# END SOLUTION: replace each ... below with the effort level named beside it

print("quick:", quick.content[0].text)
print()
print("tangled:", tangled.content[0].text)

check("effort_per_step", quick=quick, tangled=tangled)

---

## Task 3 - Check `stop_reason` before the customer sees it

In [ ]:
# ============================================================
# TASK 3 - Check stop_reason before the customer sees it
# ============================================================
#
# WHAT TO DO
#   A customer asks about two orders in one message. The store
#   sends every request with max_tokens=300. Finish deliver():
#   it reads stop_reason first and, if the reply was cut,
#   retries with more room before anything is shown.
#
# WHY IT MATTERS
#   max_tokens is a ceiling, not a target. Claude does not aim
#   for it; the reply simply stops when it gets there - here,
#   halfway through the second order. If the application ignores
#   stop_reason, the customer gets a broken answer. If it checks
#   first, it can raise the ceiling, ask for a shorter answer,
#   or retry with more room.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The third setting is max tokens".
#
# HOW TO DO IT
#   Two blanks inside deliver():
#
#       "max_tokens"   the stop_reason of a cut reply, in quotes
#       1000           the larger ceiling for the retry
# ============================================================

TWO_ORDERS = "I have two orders, ORD-12345678 and ORD-87654321. Can I return both, and how?"

first = client.messages.create(
    model=MODEL_CHAT,
    max_tokens=300,
    output_config={"effort": "low"},
    system=SYSTEM,
    messages=[{"role": "user", "content": TWO_ORDERS}],
)
print("first try:", first.stop_reason)
print("...", repr(first.content[0].text[-70:]))
print()


# BEGIN SOLUTION
def deliver(response):
    if response.stop_reason == "max_tokens":
        response = client.messages.create(
            model=MODEL_CHAT,
            max_tokens=1000,
            output_config={"effort": "low"},
            system=SYSTEM,
            messages=[{"role": "user", "content": TWO_ORDERS}],
        )
    return response.content[0].text
# SCAFFOLD: def deliver(response):
# SCAFFOLD:     if response.stop_reason == ...:   # "max_tokens" - the reply was cut
# SCAFFOLD:         response = client.messages.create(
# SCAFFOLD:             model=MODEL_CHAT,
# SCAFFOLD:             max_tokens=...,   # 1000 - room to finish
# SCAFFOLD:             output_config={"effort": "low"},
# SCAFFOLD:             system=SYSTEM,
# SCAFFOLD:             messages=[{"role": "user", "content": TWO_ORDERS}],
# SCAFFOLD:         )
# SCAFFOLD:     return response.content[0].text
# END SOLUTION: replace each ... below with the value named beside it


delivered = deliver(first)
print("what the customer sees:")
print(delivered)

check("check_before_send", first=first, delivered=delivered)

---

## Done

For every step you send to Claude, pick the model family by the step's difficulty,
speed and volume, set the effort to match, and always check `stop_reason` before a reply
reaches the customer.

One more thing the lecture mentions: older code sets `temperature`, `top_p` or `top_k`,
or prefills the start of the reply. Current Opus, Sonnet and Fable models reject these.
Try it - add `temperature=0` to any call above and run it.